# Day 87: Testing a guess with data (the "peaty layers" hypothesis)

**The guess:** the 144 layers with very high organic carbon (`flag_oc_high`) are real peaty or organic layers, not typing mistakes.

**The method:** a guess is only a story until the data agrees with it. We look for evidence for and against.

**The detective picture:** the 144 layers are the suspects. "Real peat" and "typo" are two stories. We look for clues.

> Run this notebook top to bottom. Restart the kernel first.

## 0. Setup
Python only looks in the notebook's own folder. `agronomic_functions.py` and `db_config.py` live in `csa_soil_project/`, two folders up, so we add that folder to the search path **before** importing them.

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
from agronomic_functions import build_flat, quality_flags
from db_config import run_query

## 1. Rebuild the flat table and the flags (from Days 84 to 86)
`build_flat` joins the four tables safely and checks its own row count. `quality_flags` adds six flag columns and never deletes rows.

In [ ]:
flat = build_flat("orgc", ["ph", "clay", "sand"])
flat_checked = quality_flags(flat)

print("Rows:", len(flat_checked))                    # expect 1756
print(flat_checked["flag_oc_high"].sum())            # expect 144

## 2. Bring in the clue columns
We need depth, `dataset_id` and two label columns from `wosis_orgc`. We use a **left merge** on `layer_id` (it is unique), then check that we did not lose or gain rows.

> Your own merge code from the day may look a little different. What matters is the row-count check at the end.

In [ ]:
meta = run_query('''
    SELECT layer_id, dataset_id, upper_depth, lower_depth,
           organic_surface, method_options
    FROM wosis_orgc
    WHERE country_name = 'Ghana'
''')

traced = pd.merge(flat_checked, meta, on="layer_id", how="left")

print("Original rows:", len(flat_checked))
print("Merged rows:  ", len(traced))                 # this MUST be 1756

## 3. Mini skill: `groupby`
Sort the cards into one pile per group, then look at each pile.

`traced.groupby("flag_oc_high")["upper_depth"].median()` means:
1. make one pile for `False` and one for `True`
2. look only at the `upper_depth` column
3. give me the middle value of each pile

**Median or mean?** A few odd values drag the mean. The median hardly moves.

## 4. Test 1: are the flagged layers shallow?
Peat sits in wet topsoil. If the story is true, the flagged layers should be **shallow**.

**My prediction before running:** shallower than the rest.

In [ ]:
print("Layers per group (the counts must add up to 1756):")
print(traced.groupby("flag_oc_high").size())

print("\nMedian upper_depth per group:")
print(traced.groupby("flag_oc_high")["upper_depth"].median())

print("\nMean upper_depth per group:")
print(traced.groupby("flag_oc_high")["upper_depth"].mean())

**What I found:** the mean upper depth was about 48 cm for flagged layers and about 50 cm for the others. They are almost the same, so the flagged layers are **not** shallower. The "surface peat" story is weak.

*Rule I learned: depth is a clue, not a proof. A shallow result would have fitted the story. It would not have proven it.*

## 5. Test 2: where do the 144 come from?
**Idea:** "144 out of how many?" 144 late students means one thing in a class of 150 and another in a class of 5,000.

In [ ]:
high = traced[traced["flag_oc_high"] == True]

print("Flagged layers per dataset:")
print(high["dataset_id"].value_counts())

print("\nAll layers per dataset:")
print(traced["dataset_id"].value_counts())

**What I found:** all 144 flagged layers come from `GH-GhaSP`.
- 144 of 935 layers in GH-GhaSP are flagged (144 / 935 = 15.4%)
- 0 of the 821 layers in the other datasets are flagged

The flags are not spread at random. Something is special about **one dataset**.

Three possible reasons:
1. The survey sampled organic or peaty soils on purpose.
2. It used a different lab method.
3. Different units or scale, or recording errors.

## 6. Test 3: reasons 1 and 2 (the clue columns)
Compare the 144 flagged layers with the 791 normal layers **from the same dataset**.

In [ ]:
normal = traced[(traced["dataset_id"] == "GH-GhaSP") & (traced["flag_oc_high"] == False)]
print("Normal GH-GhaSP layers:", len(normal))        # expect 791

print("\n--- Flagged (144) ---")
print(high["organic_surface"].value_counts(dropna=False))
print(high["method_options"].value_counts(dropna=False))

print("\n--- Normal (791) ---")
print(normal["organic_surface"].value_counts(dropna=False))
print(normal["method_options"].value_counts(dropna=False))

**What I found:** both groups have `organic_surface = False` and the same method (sieved at 2 mm, no spectral). So:
- reason 1 is not supported
- reason 2 is ruled out

> **Habit I learned:** always check that group counts add up to the group size (144 and 791 here). At one point I ran the same line on the wrong table and the count (144) did not match the group (791). The check caught it.

## 7. Test 4: reason 3 (is the whole dataset on a different scale?)
If GH-GhaSP recorded carbon differently, even its *normal* layers will look odd. Compare the median of every column, by dataset.

In [ ]:
print(traced.groupby("dataset_id")["orgc"].median())

print()
print(traced.groupby("dataset_id")[["orgc", "ph", "clay", "sand"]].median())

**What I found:**

| dataset | median `orgc` |
|---|---|
| AF-AfSP | 4.2 |
| **GH-GhaSP** | **103.4** |
| WD-ISIS | 2.4 |
| WD-NWAFU-SCS | 1.8 |
| WD-WISE | 3.2 |

- GH-GhaSP's median carbon is about 25 to 50 times higher than the others.
- Its `ph` is normal (5.8), so the problem is specific to the **carbon values**.
- GH-GhaSP has no `clay` or `sand` values (`NaN`).

## 8. Verdict (my own words)
1. The 144 high-carbon layers are all from GH-GhaSP, and the median carbon there is 103.4 compared with 1.8 to 4.2 in the other datasets.
2. I ruled out different organic surface sampling and different lab methods as explanations.
3. The cause is not yet confirmed. To confirm it I would check the official GH-GhaSP dataset documentation to see if they recorded carbon in a different unit.

**Note:** a unit swap may not explain all of it, because the gap (about 25 to 50 times) is not a clean 10 or 100. Recording errors are also possible.

### What I keep from today
- **Flag, don't delete.** All 144 layers stay in the table with their flag.
- **Never trust a pooled number until you have checked it by group.** One `groupby("dataset_id")` found what the table-wide mean and the `> 200` alarm could not.
- A guess that the data does not support is still a **result**.

### Next
Day 88: turn today's finding into a reusable alarm (a function that compares the median of each dataset and warns when one is far from the rest).